<a href="https://colab.research.google.com/github/FelipeMello987/GS_IA/blob/main/GS_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  MISSION CONTROL AI — ARIA
#  Global Solution 2026.1 | Prompt and Artificial Intelligence
# ============================================================

import subprocess, time, random, os

# --- instalar e subir ollama ---
os.system("curl -fsSL https://ollama.com/install.sh | sh")
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
os.system("ollama pull llama3.2:1b")

import subprocess as sp
result = sp.run(["pip", "install", "ollama", "-q"], capture_output=True)

import ollama

# --- configurações de limites operacionais ---
LIMITES = {
    "temperatura": {"ok": (0, 75),    "alerta": (75, 90),  "critico": 90},
    "energia":     {"ok": (30, 100),  "alerta": (20, 30),  "critico": 20},
    "pressao":     {"ok": (0.8, 1.2), "alerta": (0.6, 0.8),"critico": 0.6},
}

STATUS_SINAL = ["estável", "instável", "falha"]
STATUS_MODULO = ["ONLINE", "DEGRADADO", "OFFLINE"]

SYSTEM_PROMPT = """Você é ARIA (Artificial Response Intelligence for Aerospace),
assistente de controle de missão espacial. Monitora dados de telemetria e auxilia
o operador com análises, alertas e recomendações técnicas.

Parâmetros normais de operação:
- Temperatura: 0°C a 75°C | alerta > 75°C | crítico > 90°C
- Energia: acima de 30% | modo economia abaixo de 20% | crítico abaixo de 10%
- Pressão: 0.8 a 1.2 atm | alerta < 0.8 atm | crítico < 0.6 atm
- Comunicação: estável (normal), instável (alerta), falha (crítico)
- Módulos: ONLINE (normal), DEGRADADO (alerta), OFFLINE (crítico)

Responda sempre em português. Seja técnico, objetivo e conciso.
Quando receber dados de telemetria, analise e indique alertas explicitamente."""


# --- geração de telemetria simulada ---
def gerar_telemetria():
    temperatura = round(random.uniform(20, 105), 1)
    energia      = round(random.uniform(8, 98), 1)
    pressao      = round(random.uniform(0.4, 1.4), 2)
    sinal        = random.choice(STATUS_SINAL)
    modulo_a     = random.choice(STATUS_MODULO)
    modulo_b     = random.choice(STATUS_MODULO)
    return {
        "temperatura": temperatura,
        "energia":     energia,
        "pressao":     pressao,
        "sinal":       sinal,
        "modulo_a":    modulo_a,
        "modulo_b":    modulo_b,
    }


# --- classificação de status de cada parâmetro ---
def classificar(chave, valor):
    if chave == "temperatura":
        if valor > LIMITES["temperatura"]["critico"]: return "CRÍTICO"
        if valor > LIMITES["temperatura"]["alerta"][0]: return "ALERTA"
        return "NORMAL"
    if chave == "energia":
        if valor < LIMITES["energia"]["critico"]: return "CRÍTICO"
        if valor < LIMITES["energia"]["alerta"][1]: return "ALERTA"
        return "NORMAL"
    if chave == "pressao":
        if valor < LIMITES["pressao"]["critico"]: return "CRÍTICO"
        if valor < LIMITES["pressao"]["alerta"][1]: return "ALERTA"
        return "NORMAL"
    if chave == "sinal":
        return {"estável": "NORMAL", "instável": "ALERTA", "falha": "CRÍTICO"}.get(valor, "?")
    if chave in ("modulo_a", "modulo_b"):
        return {"ONLINE": "NORMAL", "DEGRADADO": "ALERTA", "OFFLINE": "CRÍTICO"}.get(valor, "?")
    return "?"


# --- lógica de tomada de decisão automática ---
def decidir_acoes(tel):
    acoes = []
    if tel["energia"] < 10:
        acoes.append("EMERGÊNCIA: desligar sistemas não essenciais imediatamente")
    elif tel["energia"] < 20:
        acoes.append("Ativar modo de economia de energia")
    if tel["temperatura"] > 90:
        acoes.append("Acionar sistema de resfriamento de emergência")
    elif tel["temperatura"] > 75:
        acoes.append("Reduzir carga dos módulos para controlar temperatura")
    if tel["pressao"] < 0.6:
        acoes.append("EMERGÊNCIA: verificar integridade do casco — pressão crítica")
    elif tel["pressao"] < 0.8:
        acoes.append("Iniciar protocolo de pressurização assistida")
    if tel["sinal"] == "falha":
        acoes.append("Ativar antena de backup — comunicação perdida")
    elif tel["sinal"] == "instável":
        acoes.append("Recalibrar sistemas de comunicação")
    if tel["modulo_a"] == "OFFLINE" or tel["modulo_b"] == "OFFLINE":
        acoes.append("Isolar módulo OFFLINE e ativar redundância")
    return acoes


# --- exibição do painel de telemetria ---
def exibir_painel(tel):
    st = {k: classificar(k, v) for k, v in tel.items()}
    indicador = {"NORMAL": "[ OK ]", "ALERTA": "[WARN]", "CRÍTICO": "[CRIT]"}

    print()
    print("=" * 56)
    print("   MISSION CONTROL AI — ARIA  |  TELEMETRIA ATUAL")
    print("=" * 56)
    print(f"  {indicador[st['temperatura']]}  Temperatura  : {tel['temperatura']}°C")
    print(f"  {indicador[st['energia']]}  Energia       : {tel['energia']}%")
    print(f"  {indicador[st['pressao']]}  Pressão       : {tel['pressao']} atm")
    print(f"  {indicador[st['sinal']]}  Comunicação   : {tel['sinal']}")
    print(f"  {indicador[st['modulo_a']]}  Módulo A      : {tel['modulo_a']}")
    print(f"  {indicador[st['modulo_b']]}  Módulo B      : {tel['modulo_b']}")
    print("-" * 56)

    acoes = decidir_acoes(tel)
    if acoes:
        print("  AÇÕES RECOMENDADAS:")
        for a in acoes:
            print(f"    >> {a}")
        print("-" * 56)


# --- chamada à IA com histórico de conversa ---
historico = []

def consultar_aria(mensagem_usuario, tel):
    contexto_tel = (
        f"\n[TELEMETRIA: Temp={tel['temperatura']}°C | "
        f"Energia={tel['energia']}% | "
        f"Pressão={tel['pressao']}atm | "
        f"Sinal={tel['sinal']} | "
        f"MódA={tel['modulo_a']} | "
        f"MódB={tel['modulo_b']}]"
    )
    historico.append({
        "role": "user",
        "content": mensagem_usuario + contexto_tel
    })

    resposta = ollama.chat(
        model="llama3.2:1b",
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + historico
    )
    texto = resposta["message"]["content"]
    historico.append({"role": "assistant", "content": texto})
    return texto


# --- loop principal do chatbot ---
def iniciar_chatbot():
    tel = gerar_telemetria()

    print()
    print("=" * 56)
    print("   MISSION CONTROL AI — ARIA")
    print("   Global Solution 2026.1 | Prompt & AI — FIAP")
    print("=" * 56)
    print("  Sistema inicializado. Telemetria carregada.")
    print("  Digite 'sair' para encerrar | 'atualizar' para nova telemetria")
    print("=" * 56)

    exibir_painel(tel)

    boas_vindas = consultar_aria("Sistema inicializado. Faça uma análise inicial dos dados de telemetria.", tel)
    print("\n  ARIA:")
    print(f"  {boas_vindas}")
    print()

    while True:
        print("-" * 56)
        entrada = input("  Operador: ").strip()

        if not entrada:
            continue

        if entrada.lower() == "sair":
            print("\n  ARIA: Encerrando sessão de monitoramento. Boa missão.")
            break

        if entrada.lower() == "atualizar":
            tel = gerar_telemetria()
            exibir_painel(tel)
            continue

        resposta = consultar_aria(entrada, tel)
        print(f"\n  ARIA: {resposta}\n")


iniciar_chatbot()


   MISSION CONTROL AI — ARIA
   Global Solution 2026.1 | Prompt & AI — FIAP
  Sistema inicializado. Telemetria carregada.
  Digite 'sair' para encerrar | 'atualizar' para nova telemetria

   MISSION CONTROL AI — ARIA  |  TELEMETRIA ATUAL
  [ OK ]  Temperatura  : 30.5°C
  [ OK ]  Energia       : 65.4%
  [ OK ]  Pressão       : 1.27 atm
  [WARN]  Comunicação   : instável
  [CRIT]  Módulo A      : OFFLINE
  [CRIT]  Módulo B      : OFFLINE
--------------------------------------------------------
  AÇÕES RECOMENDADAS:
    >> Recalibrar sistemas de comunicação
    >> Isolar módulo OFFLINE e ativar redundância
--------------------------------------------------------

  ARIA:
  Análise inicial dos dados de telemetria:

**Tempo**: O valor de temperatura está dentro do intervalo normal (0°C a 75°C). No entanto, é importante notar que o limite superior (75°C) não foi atingido.

**Energia**: A energia atual está abaixo do nível normal (30% abaixo do máximo permitido).

**Pressão**: A pressão está